In [0]:
from pyspark.sql.functions import col, to_timestamp, sum as _sum, count, countDistinct, collect_set, avg, concat_ws, row_number
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
BRONZE = "delta/bronze"
SILVER = "delta/silver"

In [0]:
orders      = spark.table("bronze.orders")
order_items = spark.table("bronze.order_items")
customers   = spark.table("bronze.customers")
products    = spark.table("bronze.products")
sellers     = spark.table("bronze.sellers")
payments    = spark.table("bronze.payments")
reviews     = spark.table("bronze.reviews")

In [0]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

In [0]:
orders_clean = orders
for c in date_cols:
    orders_clean = orders_clean.withColumn(c, to_timestamp(col(c)))

In [0]:
orders_clean = orders_clean.filter(
    col("order_id").isNotNull() & col("customer_id").isNotNull()
)

In [0]:
window_dedup = Window.partitionBy("order_id").orderBy(col("order_purchase_timestamp").desc())
orders_clean = (
    orders_clean
    .withColumn("_row", F.row_number().over(window_dedup))
    .filter(col("_row") == 1)
    .drop("_row")
)

In [0]:
orders_clean = orders_clean.filter(col("order_status").isin(["delivered", "shipped"]))

print("orders_clean OK")

orders_clean OK


In [0]:
consolidated = (
    orders_clean
    .join(order_items.drop("ingest_timestamp"),  "order_id",    "left")
    .join(customers.drop("ingest_timestamp"),    "customer_id", "left")
    .join(products.drop("ingest_timestamp"),     "product_id",  "left")
    .join(sellers.drop("ingest_timestamp"),      "seller_id",   "left")
)

In [0]:
payments_summary = (
    payments
    .groupBy("order_id")
    .agg(
        _sum("payment_value").alias("total_paid"),
        _sum("payment_installments").alias("total_installments"),
        collect_set("payment_type").alias("payment_types_array"),
    )
    .withColumn("payment_types", concat_ws(",", col("payment_types_array")))
    .drop("payment_types_array")
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
consolidated.write.mode("overwrite").format("delta").saveAsTable("silver.orders_consolidated")
payments_summary.write.mode("overwrite").format("delta").saveAsTable("silver.payments_summary")

print("[SILVER] Concluído.")

[SILVER] Concluído.


In [0]:
payments_summary.write.mode("overwrite").format("delta").saveAsTable("silver.payments_summary")

print(f"[SILVER] ✓ payments_summary salvo | linhas: {payments_summary.count()}")
print("[SILVER] Camada Silver concluída com sucesso.")

[SILVER] ✓ payments_summary salvo | linhas: 99440
[SILVER] Camada Silver concluída com sucesso.


In [0]:
spark.stop()